### Import Required Dependencies

In [ ]:
# --- Imports ---
import pandas as pd
import numpy as np
import warnings
import logging
import matplotlib.pyplot as plt
import seaborn as sns
import re
import sys
from functools import reduce



from pathlib import Path
from functools import reduce
from sklearn.preprocessing import MinMaxScaler
from collections import Counter

# --- Setup ---
sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
warnings.filterwarnings('ignore')

# --- Paths ---
data_dir = Path("data")
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

log_dir = Path("logs")
log_dir.mkdir(exist_ok=True)

image_dir = Path("outputs/images")
image_dir.mkdir(parents=True, exist_ok=True)

# Clear existing handlers
logging.getLogger().handlers.clear()

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_dir / "project.log", mode='w', encoding='utf-8'),
        logging.StreamHandler(sys.stdout)
    ]
)


print("Environment ready. Paths and logging configured.")



# Section 2 Load and Process Dataset (national and Arkansas)

## Section 2A: Load utility functions

In [ ]:
# Section 2A: Load utility functions and national datasets ----

# Import custom utility functions
from utils import (
    standardize_column_names,
    extract_key_indicators,
    filter_to_county_level,
    log_duplicate_attributes,
    clean_and_extract_year,
    nca_counties
)

data_dir = Path("data")
complete_dir = data_dir / "complete_sets"

complete_files = {
    "edu": "Education2023.csv",
    "pop": "PopulationEstimates.csv",
    "poverty": "Poverty2023.csv",
    "unemp": "Unemployment2023.csv"
}

# Load, clean, and extract year from each dataset
complete_data = {}

for key, filename in complete_files.items():
    path = complete_dir / filename
    try:
        df = pd.read_csv(path, encoding='cp1252')
        df = standardize_column_names(df)
        logging.info(f" {key} columns after cleaning: {df.columns.tolist()}")
        df = filter_to_county_level(df)
        df = clean_and_extract_year(df)

        # Rename fields if necessary
        df.rename(columns={'area_name': 'county'}, inplace=True)

        log_duplicate_attributes(df, key)
        complete_data[key] = df
        logging.info(f"Loaded {filename}: {df.shape[0]} rows")

    except Exception as e:
        logging.error(f" Failed to load {filename}: {e}")


# Confirmation message
logging.info("Utility functions from utils.py loaded successfully.")


## Section 2B: Load and process national datasets

In [ ]:
# Section 2B: Load and process national datasets

complete_dir = data_dir / "complete_sets"
complete_files = {
    "edu": "Education2023.csv",
    "pop": "PopulationEstimates.csv",
    "poverty": "Poverty2023.csv",
    "unemp": "Unemployment2023.csv"
}

state_lookup = None

# Choose your analysis year here
selected_year = '2023'

for key, filename in complete_files.items():
    path = complete_dir / filename
    try:
        # Load and standardize
        df = pd.read_csv(path, encoding='cp1252')
        df = standardize_column_names(df)
        df = filter_to_county_level(df)
        df.rename(columns={'area_name': 'county', 'fips_code': 'fips', 'fipstxt': 'fips'}, inplace=True)
        df['county'] = df['county'].str.strip().str.lower()

        # Extract year from attribute
        if 'attribute' in df.columns:
            df = clean_and_extract_year(df)

            # Only filter datasets where year tagging applies
            if key in ['pop', 'poverty', 'unemp']:
                df = df[df['attribute_year'] == selected_year]

        # Save state info once from education
        if key == 'edu':
            state_lookup = df[['county', 'state']].drop_duplicates().copy()
            state_lookup['county'] = state_lookup['county'].str.lower().str.strip()
            state_lookup['state'] = state_lookup['state'].str.upper().str.strip()

        log_duplicate_attributes(df, key)
        complete_data[key] = df
        logging.info(f" Loaded and processed {filename} with year filter: {selected_year if key != 'edu' else 'N/A'}")

    except Exception as e:
        logging.error(f" Failed to load {filename}: {e}")


### Section 2B.1 Determine most common year across all datasets

In [ ]:
## Section 2B.1: Discover Common Year Across Datasets

def get_attribute_year_counts(df):
    """Counts how often each extracted attribute_year appears."""
    if 'attribute' in df.columns:
        df = clean_and_extract_year(df)
        return Counter(df['attribute_year'].dropna().astype(str))
    return {}

year_summary = {}

# Loop through each dataset and extract year counts
for key, df in complete_data.items():
    year_counts = get_attribute_year_counts(df)
    year_summary[key] = year_counts

# Display results
print("Year coverage by dataset:")
all_years = set()

for key, counts in year_summary.items():
    print(f"\n {key.upper()}:")
    if counts:
        for year, count in sorted(counts.items()):
            print(f"  {year}: {count}")
            all_years.add(year)
    else:
        print("  No attribute_year values found.")

# Find common years across all datasets that have year values
datasets_with_years = [set(c.keys()) for c in year_summary.values() if c]
common_years = set.intersection(*datasets_with_years) if datasets_with_years else set()



print("\n Common years across all datasets with usable attribute_year:", sorted(common_years))


# Section 3 – Merge and Prepare National County-Level Dataset

# Section 4 – State & National Summary Scores